In [1]:
import os
import sys
# add the parent directory to the path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

In [2]:
import pandapower as pp
import pandapower.networks as pn
import pandas as pd
from pathlib import Path
from power_grid_model.utils import json_serialize_to_file
from power_grid_model import (
    CalculationMethod,
    CalculationType,
    ComponentType,
    PowerGridModel
)
from modules.pp_2_pgm import PandaPowerNetwork2PGM

# Convert From PandaPower Network to PowerGridModel Network

In [3]:
net = PandaPowerNetwork2PGM(net_name="cigre_mv")

input_data, extra_info = net.input_data, net.extra_info

# PGM Validation

In [4]:
from power_grid_model.validation import assert_valid_input_data

assert_valid_input_data(
    input_data, calculation_type=CalculationType.power_flow, symmetric=True)

# PGM Power Flow and Comparison with PandaPower Power Flow

In [5]:
model = PowerGridModel(input_data)

In [6]:
output_data = model.calculate_power_flow(
    symmetric=True, error_tolerance=1e-8, max_iterations=20, calculation_method=CalculationMethod.newton_raphson
)

# result dataset
print("------node result------")
print(pd.DataFrame(output_data[ComponentType.node]))

------node result------
    id  energized      u_pu              u   u_angle             p  \
0    0          1  1.025912  112850.273727 -0.008175  4.504880e+07   
1    1          1  0.987629   19752.572280 -0.647188 -1.983900e+07   
2    2          1  0.963663   19273.256820 -0.665627 -5.385557e-09   
3    3          1  0.926267   18525.336992 -0.696048 -5.017000e+05   
4    4          1  0.924394   18487.881484 -0.697881 -4.316500e+05   
5    5          1  0.923112   18462.230106 -0.699144 -7.275000e+05   
6    6          1  0.921601   18432.026751 -0.700646 -5.480500e+05   
7    7          1  0.920395   18407.906345 -0.700606 -7.650000e+04   
8    8          1  0.920680   18413.590479 -0.700420 -5.868500e+05   
9    9          1  0.919692   18393.831038 -0.701128 -5.737500e+05   
10  10          1  0.918437   18368.736969 -0.702296 -5.433000e+05   
11  11          1  0.918242   18364.839129 -0.702492 -3.298000e+05   
12  12          1  0.995890   19917.803785 -0.628337 -2.001000e+07

In [7]:
pp_net = pn.create_cigre_network_mv(with_der=False)
runpp = pp.runpp(pp_net, calculate_voltage_angles=True)
print("------pandapower result------")
print(pd.DataFrame(pp_net.res_bus))

numba cannot be imported and numba functions are disabled.
Probably the execution is slow.
Please install numba to gain a massive speedup.
(or if you prefer slow execution, set the flag numba=False to avoid this warning!)


------pandapower result------
       vm_pu  va_degree       p_mw     q_mvar
0   1.030000   0.000000 -45.045732 -16.341411
1   0.991972 -36.556771  19.839000   4.637136
2   0.968147 -37.604191   0.000000   0.000000
3   0.930961 -39.330960   0.501700   0.208882
4   0.929098 -39.434973   0.431650   0.108182
5   0.927823 -39.506622   0.727500   0.182329
6   0.926321 -39.591791   0.548050   0.137354
7   0.925122 -39.589560   0.076500   0.047410
8   0.925404 -39.578992   0.586850   0.147078
9   0.924422 -39.619167   0.573750   0.355578
10  0.923174 -39.685430   0.543300   0.161264
11  0.922980 -39.696527   0.329800   0.082656
12  1.000146 -35.487100  20.010000   4.693341
13  0.995326 -35.538237   0.034000   0.021071
14  0.992553 -35.567913   0.540050   0.257713


In [8]:
# compare node power injections from pandapower and power_grid_model

# make the injection direction consistent with PGM and convert MV to W
pp_p = -pp_net.res_bus['p_mw']*1e6
pgm_p = output_data[ComponentType.node]['p']
comparison_df = pd.DataFrame(
    {
        "pp_p": pp_p,  # convert MW to W for PandaPower Results
        "pgm_p": pgm_p,
        'diff in percent': (pp_p - pgm_p)/pgm_p*100,
        'ratio': pp_p/pgm_p
    }
)
print(comparison_df)

            pp_p         pgm_p  diff in percent     ratio
0   4.504573e+07  4.504880e+07    -6.812665e-03  0.999932
1  -1.983900e+07 -1.983900e+07    -9.388806e-14  1.000000
2  -0.000000e+00 -5.385557e-09    -1.000000e+02  0.000000
3  -5.017000e+05 -5.017000e+05    -2.874997e-11  1.000000
4  -4.316500e+05 -4.316500e+05    -6.081699e-12  1.000000
5  -7.275000e+05 -7.275000e+05    -2.109078e-11  1.000000
6  -5.480500e+05 -5.480500e+05     8.496694e-13  1.000000
7  -7.650000e+04 -7.650000e+04    -6.100391e-11  1.000000
8  -5.868500e+05 -5.868500e+05     6.096009e-11  1.000000
9  -5.737500e+05 -5.737500e+05    -8.582777e-12  1.000000
10 -5.433000e+05 -5.433000e+05     1.274933e-11  1.000000
11 -3.298000e+05 -3.298000e+05    -5.790762e-11  1.000000
12 -2.001000e+07 -2.001000e+07    -0.000000e+00  1.000000
13 -3.400000e+04 -3.400000e+04    -1.467603e-10  1.000000
14 -5.400500e+05 -5.400500e+05     9.980612e-12  1.000000


# Save the PGM Grid to a JSON file

In [9]:
notebook_dir = Path.cwd()  # get the current directory
json_serialize_to_file(notebook_dir/"cigre_mv_in_pgm.json",
                       input_data)  # save the grid to a json file